In [2]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.tree import export_graphviz
from sklearn.externals.six import StringIO
from IPython.display import Image
import pydotplus


def tree(features, target, test_size, crit="entropy", depth=None):
    x_train, x_test, y_train, y_test = train_test_split(features,
                                                        target,
                                                        test_size=test_size,
                                                        random_state=1)

    clf = DecisionTreeClassifier(max_depth=depth, criterion=crit)
    clf = clf.fit(x_train, y_train)
    pred = clf.predict(x_test)

    return clf, metrics.accuracy_score(y_test, pred)


ins_csv = "data/MotorInsuranceFraudClaimABTFull.csv"

ins_header = ["id", "type", "income", "married", "claimants",
              "injury type", "stay", "claim amount", "total claimed",
              "claims", "soft tissue", "p soft tissue", "amount received", "flag"]
ins_feature_names = ["type", "income", "claimants",
                     "injury type", "stay", "claim amount", "total claimed",
                     "claims", "soft tissue", "p soft tissue", "amount received"]

ins_dtype = {"type": "category", "married": "category", "injury type": "category", "stay": "category"}

with open(ins_csv, "r") as ins_data:
    ins = pd.read_csv(ins_data, header=0, names=ins_header, dtype=ins_dtype)
    ins = ins[ins_feature_names + ["flag"]].dropna()

    for c in ins.select_dtypes("category"):
        ins[c] = ins[c].cat.codes

    ins_features = ins[ins_feature_names]
    flag = ins.flag

    tree_data = [tree(ins_features, flag, 0.9, crit="gini"),
                 tree(ins_features, flag, 0.9),
                 tree(ins_features, flag, 0.7),
                 tree(ins_features, flag, 0.7, crit="gini"),
                 tree(ins_features, flag, 0.7, depth=1),
                 tree(ins_features, flag, 0.7, depth=2),
                 tree(ins_features, flag, 0.7, depth=5),
                 tree(ins_features, flag, 0.7, depth=10)]

    for (i, (t, a)) in enumerate(tree_data):
        dot_data = StringIO()
        export_graphviz(t, out_file=dot_data,
                        filled=True, rounded=True,
                        special_characters=True,
                        feature_names=ins_feature_names,
                        class_names=['0', '1'])
        graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
        print(str(i) + ": " + str(a))
        graph.write_png(str(i))
        Image(graph.create_png())

0: 0.9931972789115646
1: 0.9931972789115646
2: 1.0
3: 1.0
4: 1.0
5: 1.0
6: 1.0
7: 1.0
